In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 1 · Environment Setup (Shift+Enter to run)                       │
# │                                                                        │
# │  Welcome! This notebook runs GEMC directly in your browser.            │
# │  How to use:                                                           │
# │                                                                        │
# │  1. Click any grey code cell to select it                              │
# │  2. Press `Shift + Enter` to run it (or click Run in the toolbar)      │
# │  3. Run cells in order, top to bottom                                  │
# │  4. Wait for `In [*]` to become `In [1]` before running the next cell  │
# │                                                                        │
# │  Optional cells are provided to edit the code and the YAML files.      │
# │  After editing, re-run the other cells to see the changes.             │
# │                                                                        │
# │  Example: quickstart -> generated counter system.                      │
# │                                                                        │
# │  This notebook creates a small counter detector from the GEMC system   │
# │  template, builds its geometry, runs GEMC, and plots the output.       │
# │                                                                        │
# │  Import notebook helpers                                               │
# │  Create the quickstart counter files                                   │
# └────────────────────────────────────────────────────────────────────────┘

import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, display

if Path.cwd().name == "counter":
    os.chdir(Path.cwd().parent)

sys.path.insert(0, str(Path.cwd().parent))
from notebook_tools import edit

from pygemc.api.run_geometry import run_geometry

subprocess.run(["rm", "-rf", "counter"], check=True)
subprocess.run(["gemc-system-template", "-s", "counter"], check=True)
os.chdir("counter")

result = subprocess.run(["ls", "-l"], capture_output=True, text=True, check=True)
print(result.stdout)


In [ ]:
# ┌───────────────────────────────────────────────────────┐
# │  Cell 2 · Build the geometry (Shift+Enter to run)     │
# │  Run counter.py with run_geometry to create gemc.db.  │
# │  You can edit this file later; see cells below.       │
# └───────────────────────────────────────────────────────┘

print(Path("counter.py").read_text())
run_geometry("counter.py")


In [ ]:
# ┌───────────────────────────────────────────────────────────────┐
# │  Cell 3 · Run GEMC and show the event display (Shift+Enter)   │
# │  Run 100 events and display the generated GEMC PNG image.     │
# │                                                               │
# │  If GEMC crashes, just re-run this cell.                      │
# └───────────────────────────────────────────────────────────────┘

driver = '-g4view=[{driver: TOOLSSG_OFFSCREEN}]'
camera = '-g4camera=[{phi: -10*deg, theta: 250*deg}]'
light = '-g4light=[{phi: 160*deg, theta: 120*deg}]'

result = subprocess.run(
    [
        "gemc",
        "counter.yaml",
        driver,
        camera,
        light,
        "-n=100",
        "-nthreads=1",
    ],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("GEMC failed:")
    print(result.stderr)
    print("If this was an intermittent startup or display failure, re-run this cell.")
else:
    print(result.stdout)

image_file = Path("gemc_run_0.png")
if image_file.exists():
    display(Image(filename=str(image_file)))
else:
    print(f"No GEMC image found: {image_file}")



In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  Cell 4 · Plot total energy deposited (Shift+Enter to run)  │
# │  Save and display a PNG image from the analyzer histogram.  │
# └─────────────────────────────────────────────────────────────┘

from pygemc import read_output, plot_variable

plot_variable(
    read_output("counter_t0_digitized.csv", kind="csv"),
    "totEdep",
    data="digitized",
    bins=50,
    logy=True,
)



In [ ]:
# ┌─────────────────────────────────────────────────────┐
# │  Cell 5 · Show JSON output (Shift+Enter to run)     │
# │  Show the first 25 lines of counter_t0.json.        │
# └─────────────────────────────────────────────────────┘

json_file = Path("counter_t0.json")
print("".join(json_file.read_text().splitlines(True)[:25]))


In [ ]:
# ┌───────────────────────────────────────────────────────────┐
# │  Optional: Cell 6 · Edit counter.py (Shift+Enter to run)  │
# └───────────────────────────────────────────────────────────┘

edit("counter.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────┐
# │  Optional: Cell 7 · Edit geometry.py (Shift+Enter to run)  │
# └────────────────────────────────────────────────────────────┘

edit("geometry.py")


In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  Optional: Cell 8 · Edit materials.py (Shift+Enter to run)  │
# └─────────────────────────────────────────────────────────────┘

edit("materials.py")


In [ ]:
# ┌─────────────────────────────────────────────────────────────┐
# │  Optional: Cell 9 · Edit counter.yaml (Shift+Enter to run)  │
# └─────────────────────────────────────────────────────────────┘

edit("counter.yaml")
